In [1]:
from langchain.agents import create_agent
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from twelvedata import TDClient

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
twelve_data_api_key = os.getenv("TWELVE_DATA_API_KEY")
td = TDClient(apikey=twelve_data_api_key)



In [2]:
rsi_dict = {}

In [3]:
ticker_symbol = "NVDA"

In [4]:
import requests

RSI_response = requests.get(f"https://api.twelvedata.com/rsi?symbol={ticker_symbol}&interval=1day&apikey={twelve_data_api_key}")


current_RSI = RSI_response.json()["values"][-1]

In [5]:
print(RSI_response.json())

{'meta': {'symbol': 'NVDA', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'RSI - Relative Strength Index', 'series_type': 'close', 'time_period': 14}}, 'values': [{'datetime': '2026-06-05', 'rsi': '43.83642'}, {'datetime': '2026-06-04', 'rsi': '54.24632'}, {'datetime': '2026-06-03', 'rsi': '51.13961'}, {'datetime': '2026-06-02', 'rsi': '58.79015'}, {'datetime': '2026-06-01', 'rsi': '60.39106'}, {'datetime': '2026-05-29', 'rsi': '49.40963'}, {'datetime': '2026-05-28', 'rsi': '52.59496'}, {'datetime': '2026-05-27', 'rsi': '51.039989'}, {'datetime': '2026-05-26', 'rsi': '53.26204'}, {'datetime': '2026-05-22', 'rsi': '53.71362'}, {'datetime': '2026-05-21', 'rsi': '57.75770'}, {'datetime': '2026-05-20', 'rsi': '61.85447'}, {'datetime': '2026-05-19', 'rsi': '59.94933'}, {'datetime': '2026-05-18', 'rsi': '61.65906'}, {'datetime': '2026-05-15', 'rsi': '64.66334'}, {'datetim

In [6]:
rsi_values = []

for i in range(5):
    rsi_values.append(float(RSI_response.json()["values"][i]['rsi']))

In [7]:
print(rsi_values)

[43.83642, 54.24632, 51.13961, 58.79015, 60.39106]


In [8]:
def calculate_rsi_trend(rsi_values):
    """
    rsi_values should be ordered newest -> oldest.
    Uses the most recent 5 RSI values.
    """

    if len(rsi_values) < 5:
        raise ValueError("Need at least 5 RSI values")

    current_rsi = rsi_values[0]
    rsi_5_days_ago = rsi_values[4]

    slope = (current_rsi - rsi_5_days_ago) / 4

    return slope

In [9]:
def classify_rsi_trend(slope):
    if slope > 1:
        return "strongly_rising"
    elif slope > 0.3:
        return "rising"
    elif slope < -1:
        return "strongly_falling"
    elif slope < -0.3:
        return "falling"
    else:
        return "flat"

In [10]:
slope = calculate_rsi_trend(rsi_values)

print(slope)

trend = classify_rsi_trend(-1.57)

print(trend)

-4.138660000000002
strongly_falling


In [11]:
def classify_rsi_zone(current_rsi, rsi_trend):
    if current_rsi < 30 and rsi_trend == "rising":
        signal = "bullish"

    elif current_rsi > 70 and rsi_trend == "falling":
        signal = "bearish"

    else:
        signal = "neutral"

    return signal

In [12]:
rsi_zone = classify_rsi_zone(float(RSI_response.json()["values"][0]['rsi']), trend)

In [13]:
print(rsi_zone)

neutral


In [14]:
rsi_dict["current_RSI"] = float(RSI_response.json()["values"][0]['rsi'])
rsi_dict["rsi_trend"] = trend
rsi_dict["rsi_zone"] = rsi_zone

In [15]:
print(rsi_dict)

{'current_RSI': 43.83642, 'rsi_trend': 'strongly_falling', 'rsi_zone': 'neutral'}


In [123]:
moving_average_20 = requests.get(f"https://api.twelvedata.com/ma?symbol={ticker_symbol}&interval=1day&time_period=20&apikey={twelve_data_api_key}")
print(moving_average_20.json())

{'meta': {'symbol': 'NVDA', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'MA - Moving Average', 'series_type': 'close', 'time_period': 20, 'ma_type': 'SMA'}}, 'values': [{'datetime': '2026-06-05', 'ma': '219.10450'}, {'datetime': '2026-06-04', 'ma': '219.42450'}, {'datetime': '2026-06-03', 'ma': '218.88300'}, {'datetime': '2026-06-02', 'ma': '217.97050'}, {'datetime': '2026-06-01', 'ma': '216.75350'}, {'datetime': '2026-05-29', 'ma': '215.45800'}, {'datetime': '2026-05-28', 'ma': '214.87950'}, {'datetime': '2026-05-27', 'ma': '214.62950'}, {'datetime': '2026-05-26', 'ma': '214.65800'}, {'datetime': '2026-05-22', 'ma': '214.74550'}, {'datetime': '2026-05-21', 'ma': '214.39250'}, {'datetime': '2026-05-20', 'ma': '213.39900'}, {'datetime': '2026-05-19', 'ma': '212.35050'}, {'datetime': '2026-05-18', 'ma': '211.31400'}, {'datetime': '2026-05-15', 'ma': '210.30100'}, {'

In [124]:
moving_average_50 = requests.get(f"https://api.twelvedata.com/ma?symbol={ticker_symbol}&interval=1day&time_period=50&apikey={twelve_data_api_key}")
print(moving_average_50.json())

{'meta': {'symbol': 'NVDA', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'MA - Moving Average', 'series_type': 'close', 'time_period': 50, 'ma_type': 'SMA'}}, 'values': [{'datetime': '2026-06-05', 'ma': '203.44700'}, {'datetime': '2026-06-04', 'ma': '202.91860'}, {'datetime': '2026-06-03', 'ma': '202.049401'}, {'datetime': '2026-06-02', 'ma': '201.26720'}, {'datetime': '2026-06-01', 'ma': '200.26480'}, {'datetime': '2026-05-29', 'ma': '199.34880'}, {'datetime': '2026-05-28', 'ma': '198.73400'}, {'datetime': '2026-05-27', 'ma': '198.087600'}, {'datetime': '2026-05-26', 'ma': '197.50000'}, {'datetime': '2026-05-22', 'ma': '196.80780'}, {'datetime': '2026-05-21', 'ma': '196.16400'}, {'datetime': '2026-05-20', 'ma': '195.49440'}, {'datetime': '2026-05-19', 'ma': '194.72040'}, {'datetime': '2026-05-18', 'ma': '193.96120'}, {'datetime': '2026-05-15', 'ma': '193.071200'},

In [125]:
moving_average_200 = requests.get(f"https://api.twelvedata.com/ma?symbol={ticker_symbol}&interval=1day&time_period=200&apikey={twelve_data_api_key}")
print(moving_average_200.json())

{'meta': {'symbol': 'NVDA', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'MA - Moving Average', 'series_type': 'close', 'time_period': 200, 'ma_type': 'SMA'}}, 'values': [{'datetime': '2026-06-05', 'ma': '188.57310'}, {'datetime': '2026-06-04', 'ma': '188.42580'}, {'datetime': '2026-06-03', 'ma': '188.24255'}, {'datetime': '2026-06-02', 'ma': '188.071050'}, {'datetime': '2026-06-01', 'ma': '187.86705'}, {'datetime': '2026-05-29', 'ma': '187.65320'}, {'datetime': '2026-05-28', 'ma': '187.51330'}, {'datetime': '2026-05-27', 'ma': '187.35235'}, {'datetime': '2026-05-26', 'ma': '187.20285'}, {'datetime': '2026-05-22', 'ma': '187.032400'}, {'datetime': '2026-05-21', 'ma': '186.85285'}, {'datetime': '2026-05-20', 'ma': '186.64660'}, {'datetime': '2026-05-19', 'ma': '186.42925'}, {'datetime': '2026-05-18', 'ma': '186.19480'}, {'datetime': '2026-05-15', 'ma': '185.97255'},

In [127]:
current_price = td.price(symbol="NVDA").as_json()

In [128]:
print(current_price)

{'price': '205.10500'}


In [129]:
ma_dict = {}

In [130]:
ma20 = float(moving_average_20.json()["values"][0]["ma"])
ma50 = float(moving_average_50.json()["values"][0]["ma"])
ma200 = float(moving_average_200.json()["values"][0]["ma"])
price = float(current_price["price"])

In [131]:
print("ma20: ",ma20)
print("ma50: ",ma50)
print("ma200: ",ma200)
print("price: ",price)

ma20:  219.1045
ma50:  203.447
ma200:  188.5731
price:  205.105


In [132]:
price_above_ma20 = price > ma20
price_above_ma50 = price > ma50
price_above_ma200 = price > ma200

In [133]:
ma_dict["price_above_ma20"]=price_above_ma20
ma_dict["price_above_ma50"]=price_above_ma50
ma_dict["price_above_ma200"]=price_above_ma200

In [134]:
print(ma_dict)

{'price_above_ma20': False, 'price_above_ma50': True, 'price_above_ma200': True}


In [135]:
def ma_alignment_classification(ma20, ma50, ma200):
    if ma20 > ma50 > ma200:
        alignment = "bullish"
    elif ma20 < ma50 < ma200:
        alignment = "bearish"
    else:
        alignment = "mixed"

    return alignment

In [136]:
ma_dict["alignment"] = ma_alignment_classification(ma20, ma50, ma200)

In [137]:
print(ma_dict["alignment"])

bullish


In [138]:
ma20_values = moving_average_20.json()["values"]
current_ma20 = float(ma20_values[0]["ma"])
past_ma20 = float(ma20_values[4]["ma"])
ma20_slope = (current_ma20 - past_ma20) / 4


In [139]:
ma50_values = moving_average_50.json()["values"]
current_ma50 = float(ma50_values[0]["ma"])
past_ma50 = float(ma50_values[4]["ma"])
ma50_slope = (current_ma50 - past_ma50) / 4


In [140]:
ma200_values = moving_average_200.json()["values"]
current_ma200 = float(ma200_values[0]["ma"])
past_ma200 = float(ma200_values[4]["ma"])
ma200_slope = (current_ma200 - past_ma200) / 4


In [141]:
pct_above_ma20 = ((price - ma20) / ma20) * 100
pct_above_ma50 = ((price - ma50) / ma50) * 100
pct_above_ma200 = ((price - ma200) / ma200) * 100

In [142]:
ma50_values = moving_average_50.json()["values"]
ma200_values = moving_average_200.json()["values"]

current_ma50 = float(ma50_values[0]["ma"])
previous_ma50 = float(ma50_values[1]["ma"])

current_ma200 = float(ma200_values[0]["ma"])
previous_ma200 = float(ma200_values[1]["ma"])

golden_cross_today = (
    current_ma50 > current_ma200 and
    previous_ma50 <= previous_ma200
)

death_cross_today = (
    current_ma50 < current_ma200 and
    previous_ma50 >= previous_ma200
)

In [143]:
print(ma_dict)

{'price_above_ma20': False, 'price_above_ma50': True, 'price_above_ma200': True, 'alignment': 'bullish'}


In [144]:
ma_dict["pct_above_ma20"] = round(((price - ma20) / ma20) * 100, 2)
ma_dict["pct_above_ma50"] = round(((price - ma50) / ma50) * 100, 2)
ma_dict["pct_above_ma200"] = round(((price - ma200) / ma200) * 100, 2)


ma_dict["ma20_slope"] = round(ma20_slope, 3)
ma_dict["ma50_slope"] = round(ma50_slope, 3)
ma_dict["ma200_slope"] = round(ma200_slope, 3)

ma_dict["golden_cross"] = golden_cross_today

In [145]:
print(ma_dict)

{'price_above_ma20': False, 'price_above_ma50': True, 'price_above_ma200': True, 'alignment': 'bullish', 'pct_above_ma20': -6.39, 'pct_above_ma50': 0.81, 'pct_above_ma200': 8.77, 'ma20_slope': 0.588, 'ma50_slope': 0.796, 'ma200_slope': 0.177, 'golden_cross': False}


In [146]:
macd_response = requests.get(f"https://api.twelvedata.com/macd?symbol={ticker_symbol}&interval=1day&apikey={twelve_data_api_key}")
print(macd_response.json())

{'meta': {'symbol': 'NVDA', 'interval': '1day', 'currency': 'USD', 'exchange_timezone': 'America/New_York', 'exchange': 'NASDAQ', 'mic_code': 'XNGS', 'type': 'Common Stock', 'indicator': {'name': 'MACD - Moving Average Convergence Divergence', 'series_type': 'close', 'fast_period': 12, 'slow_period': 26, 'signal_period': 9}}, 'values': [{'datetime': '2026-06-05', 'macd': '2.29978', 'macd_signal': '4.25901', 'macd_hist': '-1.95923'}, {'datetime': '2026-06-04', 'macd': '3.57005', 'macd_signal': '4.74882', 'macd_hist': '-1.17877'}, {'datetime': '2026-06-03', 'macd': '3.75879', 'macd_signal': '5.043514', 'macd_hist': '-1.28472'}, {'datetime': '2026-06-02', 'macd': '4.34313', 'macd_signal': '5.36469', 'macd_hist': '-1.021562'}, {'datetime': '2026-06-01', 'macd': '4.20412', 'macd_signal': '5.62008', 'macd_hist': '-1.41597'}, {'datetime': '2026-05-29', 'macd': '3.80873', 'macd_signal': '5.97408', 'macd_hist': '-2.16535'}, {'datetime': '2026-05-28', 'macd': '4.59471', 'macd_signal': '6.51541',

In [147]:
macd_data = macd_response.json()["values"]

In [148]:
current_macd = float(macd_data[0]["macd"])
current_signal = float(macd_data[0]["macd_signal"])

if current_macd > current_signal:
    macd_position = "above_signal"
elif current_macd < current_signal:
    macd_position = "below_signal"
else:
    macd_position = "at_signal"

In [149]:
current_macd = float(macd_data[0]["macd"])
current_signal = float(macd_data[0]["macd_signal"])

previous_macd = float(macd_data[1]["macd"])
previous_signal = float(macd_data[1]["macd_signal"])

In [150]:
if (
    current_macd > current_signal and
    previous_macd <= previous_signal
):
    macd_crossover = "bullish"

elif (
    current_macd < current_signal and
    previous_macd >= previous_signal
):
    macd_crossover = "bearish"

else:
    macd_crossover = "none"

In [151]:
macd_dict = {}


In [152]:
macd_dict["macd_position"] = macd_position
macd_dict["macd_crossover"] = macd_crossover

In [153]:
macd_dict

{'macd_position': 'below_signal', 'macd_crossover': 'none'}

In [154]:
rsi_dict

{'current_RSI': 43.83642,
 'rsi_trend': 'strongly_falling',
 'rsi_zone': 'neutral'}